# Reconhecimento de Gestos com Visualização Moderna

Este notebook utiliza as **MediaPipe Tasks Drawing Utilities** oficiais para exibir os marcos das mãos de forma colorida e segmentada.

In [1]:
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import math
import os
import requests
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# IMPORTAÇÃO MODERNA (API DE TASKS)
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
from mediapipe.tasks.python.vision import drawing_styles as mp_drawing_styles
from mediapipe.tasks.python.vision.hand_landmarker import HandLandmarksConnections as mp_hands

print(f"MediaPipe: {mp.__version__}")

MediaPipe: 0.10.33


## Funções de Exibição (Estilo MediaPipe Guide)

In [2]:
def display_one_image(image, title, subplot, titlesize=16):
    """Displays one image along with the predicted category name and score."""
    plt.subplot(*subplot)
    plt.imshow(image)
    if len(title) > 0:
        plt.title(title, fontsize=int(titlesize), color='black', fontdict={'verticalalignment':'center'}, pad=int(titlesize/1.5))
    return (subplot[0], subplot[1], subplot[2]+1)

def display_batch_of_images_with_gestures_and_hand_landmarks(images, results):
    """Displays a batch of images with the gesture category and its score along with the hand landmarks."""
    # Images and labels.
    images = [image.numpy_view() for image in images]
    # Note: 'results' expected as [(top_gesture, multi_hand_landmarks), ...]
    gestures = [top_gesture for (top_gesture, _) in results]
    multi_hand_landmarks_list = [multi_hand_landmarks for (_, multi_hand_landmarks) in results]

    rows = int(math.sqrt(len(images)))
    cols = len(images) // rows if rows > 0 else 1
    if rows * cols < len(images): cols += 1

    FIGSIZE = 13.0
    SPACING = 0.1
    subplot=(rows,cols, 1)
    plt.figure(figsize=(FIGSIZE, FIGSIZE))

    for i, (image, gesture) in enumerate(zip(images, gestures)):
        title = f"{gesture.category_name} ({gesture.score:.2f})"
        dynamic_titlesize = FIGSIZE*SPACING/max(rows,cols) * 40 + 3
        annotated_image = image.copy()

        for hand_landmarks in multi_hand_landmarks_list[i]:
          mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

        subplot = display_one_image(annotated_image, title, subplot, titlesize=dynamic_titlesize)

## Webcam em Tempo Real com Estilo Moderno

In [4]:
def draw_for_webcam(frame, detection_result):
    if not detection_result or not detection_result.hand_landmarks:
        return frame
    
    annotated_image = frame.copy()
    h, w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(detection_result.hand_landmarks):
        # Desenho oficial colorido
        mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

        # Legenda customizada
        hand_label = detection_result.handedness[idx][0].category_name if idx < len(detection_result.handedness) else "?"
        gesture_name = detection_result.gestures[idx][0].category_name if idx < len(detection_result.gestures) and detection_result.gestures[idx] else "Nenhum"
        
        x_min = int(min([lm.x for lm in hand_landmarks]) * w)
        y_min = int(min([lm.y for lm in hand_landmarks]) * h)
        cv2.putText(annotated_image, f"{hand_label}: {gesture_name}", (x_min, y_min - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    
    return annotated_image

model_path = "gesture_recognizer.task"
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options, running_mode=vision.RunningMode.VIDEO, num_hands=2
)

with vision.GestureRecognizer.create_from_options(options) as recognizer:
    cap = cv2.VideoCapture(0)
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        recognition_result = recognizer.recognize_for_video(mp_image, int(cv2.getTickCount() / cv2.getTickFrequency() * 1000))
        
        annotated_frame = draw_for_webcam(frame, recognition_result)
        cv2.imshow('Modern MediaPipe Recognition', annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()